# 10+1-dimensional heat equation: TT-PINN with Adam (first-order baseline)

Companion notebook to the manuscript, Section *The $10+1$-dimensional heat equation*.
It reproduces the TT-Adam baseline, panel (b) of the heat ablation figure. Problem,
TT ansatz, collocation, soft-constraint weights, initialization and plots are identical to the
Gauss–Newton notebook (`Heat_TT_GaussNewton_ablation.ipynb`); only the parameter update differs.

## Problem

$$
\partial_t u-\kappa\Delta_x u=0\quad\text{in }(0,1)^{d_x}\times(0,1),
\qquad u(x,0)=u_0(x)=\sum_{m=1}^{d_x}\sin(2\pi x_m),
\qquad d_x=10,\ \kappa=\tfrac14 .
$$

The Dirichlet data are the trace of the exact solution
$u^\star(x,t)=e^{-4\pi^2\kappa t}\sum_{m}\sin(2\pi x_m)$ (`BOUNDARY_TARGET="exact"`). Boundary and
initial conditions are imposed softly through the loss

$$
\mathcal L(u)=\tfrac12\|\partial_tu-\kappa\Delta_xu\|^2
+\frac{\lambda_{\rm b}}{2}\,\frac{1}{2d_x}\sum_{s=1}^{d_x}\sum_{\varepsilon\in\{0,1\}}\|u-u^\star\|^2_{\Gamma_{s,\varepsilon}}
+\frac{\lambda_0}{2}\|u(\cdot,0)-u_0\|^2,
\qquad \lambda_{\rm b}=10,\ \lambda_0=5,
$$

where $\Gamma_{s,\varepsilon}$ is the face $\{x_s=\varepsilon\}$.

## Method

- **TT-PINN.** Coordinates $z=(x_1,\dots,x_{10},t)$, $d=11$, TT ranks $(1,2,\dots,2,1)$. Each core is
  a $\tanh$ network $1\to128\to128\to128\to128\to r_{m-1}r_m$, giving $P=552\,872$ parameters, with
  near-identity initialization.
- **Collocation.** One factor grid of $n=32$ uniformly drawn points per coordinate, fixed during
  training, plus the endpoints $0$ and $1$ used by the face and initial terms. Neither the interior
  tensor grid nor the face grids are formed; all quantities come from one-dimensional TT contractions.
- **Gradient.** The loss gradient is evaluated in the same compressed TT form as the
  Gauss–Newton quantities, $\nabla\widehat{\mathcal L}(\theta)=\mathsf C^\top\rho_{\rm c}$, and is
  checked against reverse-mode automatic differentiation.
- **Optimizer.** Adam with $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$ and a learning rate
  decaying exponentially from $10^{-3}$ to $10^{-5}$ over 50,000 iterations.
- **Ablation.** 10 seeds; seeds, factor grids and initial TT coincide with the Gauss–Newton
  ablation. The relative $L^2$ error is evaluated every 100 iterations on midpoint grids with 256
  points per coordinate; compilation and error evaluation are excluded from the timings.

**Notebook layout.** Engine $\to$ hyperparameters $\to$ one training run with diagnostics and
solution plots $\to$ seed ablation, figures and export (CSV table, JSON histories).

## TT engine

Problem definition, TT ansatz, factor-grid contractions, the compressed gradient and the Adam
trainer. The Gauss–Newton assembly, linear solvers and line search are not needed here. Main correspondences with the manuscript:

| manuscript | code |
|---|---|
| local operators $(\ell_s,\gamma_s)$ | `run.is_time`, `model._gammas()` |
| compressed dimension $N_{\rm eff}$ | `model.N_eff` (asserted against `heat_N_eff`) |
| residual block cores (interior) and value cores (traces) | `model.class_cores`, `model.insertions` |
| residual classes: interior, $2d_x$ faces, $t=0$, with weights | `model.quad_weights`, `model.class_omega` |
| compressed gradient $\mathsf C^\top\rho_{\rm c}$ | `model.loss_and_gradient` |

In [ ]:
from dataclasses import dataclass, asdict, replace
from functools import partial
import time, json

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import random, vmap, jacfwd, lax, jit
import numpy as np
import matplotlib.pyplot as plt

PI = jnp.pi
tree_map = jax.tree_util.tree_map

# Logarithmic line-search schedule


@dataclass(frozen=True)
class Config:
    # Global choice.
    format: str = "tt"                  # TT-PINN only

    # Heat equation and collocation.
    spatial_dim: int = 10                # total input dimension is spatial_dim + 1
    diffusivity: float = 0.25
    boundary_weight: float = 1.0         # weight for the averaged spatial-boundary residual
    initial_weight: float = 1.0          # weight for the initial-condition residual
    boundary_target: str = "exact"       # "exact" or "zero"
    n_per_axis: int = 16                 # interior factor-grid points; endpoints are added
    iterations: int = 200
    error_every: int = 5
    seed: int = 0
    n_err: int = 256
    resample_each_iteration: bool = False

    # TT-SPINN setup.
    tt_rank: int = 2
    tt_width: int = 64
    tt_hidden: int = 2
    tt_init_mode: str = "near_identity"   # "near_identity" or "random"
    tt_identity_epsilon: float = 0.30      # size of the input-dependent perturbation around I_r
    tt_global_scale: bool = True           # one global amplitude instead of per-core RMS scales

    # Adam (exponential learning-rate decay from adam_lr to adam_lr_final).
    adam_lr: float = 1e-3
    adam_lr_final: float = 1e-5          # = adam_lr for a constant rate
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_eps: float = 1e-8

    @property
    def d(self):
        return self.spatial_dim + 1

    @property
    def active_rank(self):
        return self.tt_rank

    @property
    def active_width(self):
        return self.tt_width

    @property
    def active_hidden(self):
        return self.tt_hidden


def tt_ranks(d, r):
    """TT ranks (r_0, ..., r_d) with r_0 = r_d = 1."""
    return (1,) + (int(r),) * (d - 1) + (1,)


def heat_N_eff(d, n_s, rank):
    """N_eff^ch = 2 sum_s n_s r_{s-1} r_s with Lambda_s = {0, ell_s}, r_0 = r_d = 1,
    and n_s including the two endpoint nodes."""
    ranks = tt_ranks(d, rank)
    return int(sum(2 * n_s * ranks[s - 1] * ranks[s] for s in range(1, d + 1)))


@dataclass(frozen=True)
class SlotRun:
    start: int
    stop: int
    is_time: bool              # s = d (time slot): ell_s = 1, gamma_s = 1; else ell_s = 2, gamma_s = -kappa
    k: int                     # k_s
    units: tuple               # active entries a_s, flat in the padded c x c core
    n: int                     # n_s (interior points + 2 endpoints)
    P_slot: int
    N_offset: int
    P_offset: int
    L: int = 2                 # |Lambda_s| = |{0, ell_s}|

    @property
    def length(self):
        return self.stop - self.start

    @property
    def N_slot(self):
        return self.L * self.n * self.k


def build_slot_runs(d, c, n, widths):
    hidden = 0
    sizes = (1,) + tuple(widths)
    for a, b in zip(sizes[:-1], sizes[1:]):
        hidden += a * b + b
    w_in = widths[-1] if widths else 1
    ranks = tt_ranks(d, c)
    signatures = [(s == d - 1, tuple(a * c + b for a in range(ranks[s]) for b in range(ranks[s + 1])))
                  for s in range(d)]
    runs, s, N_off, P_off = [], 0, 0, 0
    while s < d:
        e = s
        while e < d and signatures[e] == signatures[s]:
            e += 1
        is_time, units = signatures[s]
        k = len(units)
        run = SlotRun(start=s, stop=e, is_time=is_time, k=k, units=units, n=n,
                      P_slot=hidden + w_in * k + k, N_offset=N_off, P_offset=P_off)
        runs.append(run)
        N_off += run.length * run.N_slot
        P_off += run.length * run.P_slot
        s = e
    return tuple(runs)


class Model:
    """TT-PINN engine for the heat equation with soft constraints.

    Notation (manuscript -> code), z = (x_1, ..., x_{d_x}, t), d = d_x + 1:
      (ell_s, gamma_s) = (2, -kappa) for s <= d_x and (1, 1) for s = d
      local channels Lambda_s = {0, ell_s}: (G_s, d^{ell_s} G_s)          -> local_channels (len, n_s, 2, k_s)
      block cores B_m^h = [[G_m, gamma_m d^{ell_m} G_m], [0, G_m]]        -> operator block cores
      insertions P_{s,0,a} = diag(M_a, M_a), P_{s,ell,a} = gamma_s [[0, M_a], [0, 0]]
      trace residuals: value direction matrices Psi^val (value chains), same C^ch
      C_s^ch = col(H_{s,0}, H_{s,ell_s}), rows (l, i_s, a_s)              -> Cs
      Upsilon = sum_c omega_c Upsilon_c, rho_c = sum_c omega_c rho_{c}  (interior, 2 d_x faces, t = 0)
      compressed gradient  grad L = C^T rho_c (C applied as one VJP)      -> loss_and_gradient
    N_eff^ch = 2 sum_s n_s k_s, where n_s = N_PER_AXIS + 2 includes the endpoints.
    """

    def __init__(self, cfg: Config):
        fmt = cfg.format.lower()
        if fmt != "tt":
            raise ValueError("this notebook implements the TT-PINN only (cfg.format='tt').")
        if cfg.boundary_target.lower() not in {"exact", "zero"}:
            raise ValueError("cfg.boundary_target must be 'exact' or 'zero'.")
        if cfg.tt_init_mode.lower() not in {"near_identity", "random"}:
            raise ValueError("cfg.tt_init_mode must be 'near_identity' or 'random'.")
        if not (0.0 < cfg.adam_lr_final <= cfg.adam_lr):
            raise ValueError("require 0 < adam_lr_final <= adam_lr.")
        if cfg.spatial_dim < 1 or cfg.n_per_axis < 1 or cfg.diffusivity <= 0.0:
            raise ValueError("spatial_dim, n_per_axis and diffusivity must be positive.")
        if cfg.boundary_weight < 0.0 or cfg.initial_weight < 0.0:
            raise ValueError("soft-constraint weights must be nonnegative.")

        self.cfg = cfg
        self.format = fmt
        self.dx = cfg.spatial_dim
        self.time_slot = self.dx
        self.d = cfg.d
        self.kappa = float(cfg.diffusivity)
        self.decay = float(4.0 * np.pi**2 * self.kappa)
        self.n_grid = cfg.n_per_axis + 2
        self.zero_index = cfg.n_per_axis
        self.one_index = cfg.n_per_axis + 1

        c = cfg.active_rank
        self.c = c
        self.q = 2 * c
        self.RF = self.dx                           # CP rank of u* (target chain)
        self.C_target = jnp.ones((self.RF,))
        self.widths = (cfg.active_width,) * cfg.active_hidden
        self.runs = build_slot_runs(self.d, c, self.n_grid, self.widths)
        self.ranks = tt_ranks(self.d, c)
        self.P = sum(r.length * r.P_slot for r in self.runs)
        self.P_slots = tuple(r.P_slot for r in self.runs for _ in range(r.length))
        self.P_axis = max(self.P_slots)
        self.k_slots = tuple(r.k for r in self.runs for _ in range(r.length))
        self.N_slots = tuple(r.N_slot for r in self.runs for _ in range(r.length))
        self.N_eff = sum(self.N_slots)
        assert self.N_eff == heat_N_eff(self.d, self.n_grid, c)
        self.sizes = (1,) + self.widths + (max(r.k for r in self.runs),)
        self._layouts = {}
        self.matrix_units = jnp.eye(c * c).reshape(c * c, c, c)

        self.lvec = jnp.eye(c)[0]
        self.rvec = jnp.eye(c)[0]
        zc = jnp.zeros((c,))
        self.op_left = jnp.concatenate([self.lvec, zc])
        self.op_right = jnp.concatenate([zc, self.rvec])      # operator chain [1 0] B ... B [0 1]^T
        self.val_right = jnp.concatenate([self.rvec, zc])     # value chain in the same 2c embedding
        self.ones_K = jnp.ones((self.RF,))
        self._build_residual_classes()

    # ---------------- residual classes and quadratures ----------------
    def _build_residual_classes(self):
        """Interior, the 2 d_x spatial faces and t = 0 on common augmented factor grids."""
        n, ng, d, dx = self.cfg.n_per_axis, self.n_grid, self.d, self.dx
        base = jnp.concatenate([jnp.full((n,), 1.0 / n), jnp.zeros((2,))])
        delta0 = jnp.zeros((ng,)).at[self.zero_index].set(1.0)
        delta1 = jnp.zeros((ng,)).at[self.one_index].set(1.0)
        interior = jnp.tile(base[None, :], (d, 1))
        exact = self.cfg.boundary_target.lower() == "exact"

        weights, value_class, omega, tau, names = [interior], [0.0], [1.0], [0.0], ["interior"]
        face_weight = self.cfg.boundary_weight / float(2 * dx)
        for s in range(dx):
            for side, delta in ((0, delta0), (1, delta1)):
                weights.append(interior.at[s].set(delta))
                value_class.append(1.0)
                omega.append(face_weight)
                tau.append(1.0 if exact else 0.0)
                names.append(f"boundary_x{s + 1}_{side}")
        weights.append(interior.at[self.time_slot].set(delta0))
        value_class.append(1.0)
        omega.append(self.cfg.initial_weight)
        tau.append(1.0)
        names.append("initial_t0")

        self.quad_weights = jnp.stack(weights)                  # (n_classes, d, n_s)
        self.class_is_value = jnp.asarray(value_class)          # 0: operator residual, 1: trace residual
        self.class_omega = jnp.asarray(omega)                   # loss weights omega_c
        self.class_tau = jnp.asarray(tau)                       # target present in the residual
        self.class_names = tuple(names)
        self.n_classes = len(names)
        self.initial_class = self.n_classes - 1
        v = self.class_is_value[:, None]
        self.class_right = (1.0 - v) * self.op_right[None, :] + v * self.val_right[None, :]

    def sample_grids(self, key):
        interior = random.uniform(key, (self.d, self.cfg.n_per_axis))
        endpoints = jnp.tile(jnp.array([0.0, 1.0])[None, :], (self.d, 1))
        return jnp.concatenate([interior, endpoints], axis=1)

    # ---------------- networks ----------------
    def _layout(self, k):
        if k not in self._layouts:
            sizes = (1,) + self.widths + (k,)
            layers = tuple(zip(sizes[:-1], sizes[1:]))
            off = [0]
            for m_in, n_out in layers:
                off.append(off[-1] + n_out * m_in + n_out)
            self._layouts[k] = (sizes, layers, tuple(off))
        return self._layouts[k]

    def network_sizes(self, run):
        return self._layout(run.k)[0]

    def mlp(self, theta_slot, x, k):
        _, layers, offsets = self._layout(k)
        z = jnp.atleast_1d(x)
        for ell, (m_in, n_out) in enumerate(layers):
            o = offsets[ell]
            w = lax.dynamic_slice(theta_slot, (o,), (n_out * m_in,)).reshape((n_out, m_in))
            b = lax.dynamic_slice(theta_slot, (o + n_out * m_in,), (n_out,))
            z = w @ z + b
            if ell < len(layers) - 1:
                z = jnp.tanh(z)
        return z

    def _channels(self, theta_slot, s, scale, run):
        """(G_s, d^{ell_s} G_s)(z): shape (2, k)."""
        f = lambda t: scale * self.mlp(theta_slot, t, run.k)
        value, first = jax.jvp(f, (s,), (1.0,))
        if run.is_time:
            return jnp.stack([value, first])
        d1 = lambda t: jax.jvp(f, (t,), (1.0,))[1]
        _, second = jax.jvp(d1, (s,), (1.0,))
        return jnp.stack([value, second])

    def _channels_and_H(self, theta_slot, s, scale, run):
        vals, pull = jax.vjp(lambda th: self._channels(th, s, scale, run).reshape(-1), theta_slot)
        H = vmap(lambda e: pull(e)[0])(jnp.eye(2 * run.k, dtype=theta_slot.dtype))
        return vals.reshape(2, run.k), H.reshape(2, run.k, -1)

    def _pad(self, local, run):
        c = self.c
        flat = jnp.zeros(local.shape[:-1] + (c * c,), dtype=local.dtype)
        flat = flat.at[..., jnp.asarray(run.units)].set(local)
        return flat.reshape(local.shape[:-1] + (c, c))

    def core_values(self, theta, grids, scales):
        """Padded value cores G_s(z): (d, n, c, c) on arbitrary per-slot grids."""
        out = []
        for run, th in zip(self.runs, theta):
            sl = slice(run.start, run.stop)
            f = lambda t, zs, sc: vmap(lambda z: scale_mlp(self, t, z, sc, run.k))(zs)
            out.append(self._pad(vmap(f)(th, grids[sl], scales[sl]), run))
        return jnp.concatenate(out, axis=0)

    def local_channels(self, theta, grids, scales):
        """Unpadded local channels per run: tuple of (len, n, 2, k_s)."""
        out = []
        for run, th in zip(self.runs, theta):
            sl = slice(run.start, run.stop)
            f = lambda t, zs, sc: vmap(lambda z: self._channels(t, z, sc, run))(zs)
            out.append(vmap(f)(th, grids[sl], scales[sl]))
        return tuple(out)

    def pad_channels(self, local):
        return jnp.concatenate([self._pad(loc, run) for run, loc in zip(self.runs, local)], axis=0)

    def core_channels(self, theta, grids, scales):
        """Padded (G_s, d^{ell_s} G_s): (d, n, 2, c, c)."""
        out = []
        for run, th in zip(self.runs, theta):
            sl = slice(run.start, run.stop)
            f = lambda t, zs, sc: vmap(lambda z: self._channels(t, z, sc, run))(zs)
            out.append(self._pad(vmap(f)(th, grids[sl], scales[sl]), run))
        return jnp.concatenate(out, axis=0)

    def core_channels_and_C(self, theta, grids, scales):
        cores, Cs = [], []
        for run, th in zip(self.runs, theta):
            sl = slice(run.start, run.stop)
            f = lambda t, zs, sc: vmap(lambda z: self._channels_and_H(t, z, sc, run))(zs)
            vals, H = vmap(f)(th, grids[sl], scales[sl])
            cores.append(self._pad(vals, run))
            H = jnp.transpose(H, (0, 2, 1, 3, 4))                 # (len, 2, n, k, P)
            Cs.append(H.reshape(run.length, run.N_slot, run.P_slot))
        return jnp.concatenate(cores, axis=0), tuple(Cs)

    def init_theta(self, key):
        """Same random stream as the previous notebook; TT boundary cores keep
        only their active outputs (identical GN trajectories for a given seed)."""
        c = self.c
        near_identity = self.cfg.tt_init_mode.lower() == "near_identity"
        Q = c * c
        _, full_layers, full_offsets = self._layout(Q)

        def one(k):
            parts = []
            layer_keys = random.split(k, len(full_layers))
            for ell, (kk, (m_in, n_out)) in enumerate(zip(layer_keys, full_layers)):
                kw, kb = random.split(kk)
                is_output = ell == len(full_layers) - 1
                if near_identity and is_output:
                    v = self.cfg.tt_identity_epsilon * jnp.sqrt(1.0 / m_in)
                    w = random.uniform(kw, (n_out, m_in), minval=-v, maxval=v)
                    b = jnp.eye(c, dtype=w.dtype).reshape(-1)
                else:
                    v = jnp.sqrt(1.0 / m_in)
                    w = random.uniform(kw, (n_out, m_in), minval=-v, maxval=v)
                    b = random.uniform(kb, (n_out,), minval=-v, maxval=v)
                parts.extend([w.ravel(), b])
            return jnp.concatenate(parts)

        full = vmap(one)(random.split(key, self.d))
        n_in = full_layers[-1][0]
        o = full_offsets[-2]
        theta = []
        for run in self.runs:
            active = np.asarray(run.units)
            idx = np.concatenate([
                np.arange(o),
                (o + active[:, None] * n_in + np.arange(n_in)[None, :]).ravel(),
                o + Q * n_in + active,
            ])
            theta.append(full[run.start:run.stop][:, idx])
        return tuple(theta)

    # ---------------- exact solution ----------------
    def target_factors(self, grids):
        """Factors of u* = exp(-4 pi^2 kappa t) sum_rho sin(2 pi x_rho): (d, n, K)."""
        d, n = grids.shape
        T = jnp.ones((d, n, self.RF))
        idx = jnp.arange(self.dx)
        T = T.at[idx, :, idx].set(jnp.sin(2.0 * PI * grids[:self.dx]))
        time_factor = jnp.exp(-self.decay * grids[self.time_slot])
        return T.at[self.time_slot].set(jnp.tile(time_factor[:, None], (1, self.RF)))

    def target_values(self, grids, coeffs=None):
        coeffs = self.C_target if coeffs is None else coeffs
        return self.target_factors(grids).at[0].multiply(coeffs[None, :])

    # ---------------- block chains ----------------
    def _gammas(self):
        return jnp.full((self.d,), -self.kappa).at[self.time_slot].set(1.0)

    def class_cores(self, cores):
        """Operator cores B^h (interior class) and value cores diag(G, G) (trace classes),
        stacked per class: (n_classes, d, n, 2c, 2c)."""
        G, DG = cores[:, :, 0], cores[:, :, 1]
        g = self._gammas()[:, None, None, None]
        Z = jnp.zeros_like(G)
        Bop = jnp.concatenate([jnp.concatenate([G, g * DG], -1), jnp.concatenate([Z, G], -1)], -2)
        Bval = jnp.concatenate([jnp.concatenate([G, Z], -1), jnp.concatenate([Z, G], -1)], -2)
        v = self.class_is_value[:, None, None, None, None]
        return (1.0 - v) * Bop[None] + v * Bval[None]

    def insertions(self, run):
        """P_{s,0,a} = diag(M_a, M_a), P_{s,ell,a} = gamma_s [[0, M_a], [0, 0]]: (2, k, q, q)."""
        gamma = 1.0 if run.is_time else -self.kappa
        M = self.matrix_units[jnp.asarray(run.units)]
        Z = jnp.zeros_like(M)
        P0 = jnp.concatenate([jnp.concatenate([M, Z], -1), jnp.concatenate([Z, M], -1)], -2)
        P1 = gamma * jnp.concatenate([jnp.concatenate([Z, M], -1), jnp.concatenate([Z, Z], -1)], -2)
        return jnp.stack([P0, P1])

    def _transfers(self, Gc):
        """T[x,m,a,b,c,e] = sum_i w_{x,m}(i) Gc[x,m,i,a,c] Gc[x,m,i,b,e]."""
        return jnp.einsum("xmi,xmiac,xmibe->xmabce", self.quad_weights, Gc, Gc)

    def _env_mm(self, T):
        """Per-class model/model environments from the transfers."""
        def one(Tx, right):
            L0, R0 = jnp.outer(self.op_left, self.op_left), jnp.outer(right, right)
            Lf, Lb = lax.scan(lambda L, Tm: (jnp.einsum("ab,abce->ce", L, Tm), L), L0, Tx)
            _, Ra = lax.scan(lambda R, Tm: (jnp.einsum("abce,ce->ab", Tm, R), R), R0, Tx, reverse=True)
            return Lb, Ra, jnp.einsum("ac,a,c->", Lf, right, right)
        return vmap(one)(T, self.class_right)

    def _env_mf(self, Gc, S):
        """Per-class model/target environments; S: diagonal CP target chain (d, n, K)."""
        def one(Gx, Wx, right):
            L0, R0 = jnp.outer(self.op_left, self.ones_K), jnp.outer(right, self.ones_K)
            ls = lambda L, x: (jnp.einsum("ac,iab,ic,i->bc", L, x[0], x[1], x[2]), L)
            rs = lambda R, x: (jnp.einsum("iab,ic,i,bc->ac", x[0], x[1], x[2], R), R)
            Lf, Lb = lax.scan(ls, L0, (Gx, S, Wx))
            _, Ra = lax.scan(rs, R0, (Gx, S, Wx), reverse=True)
            return Lb, Ra, jnp.einsum("ac,a,c->", Lf, right, self.ones_K)
        return vmap(one)(Gc, self.quad_weights, self.class_right)

    def _quad_ff(self, S):
        """<g, g>_c for every class (diagonal CP target chain)."""
        gram = jnp.einsum("xmi,mia,mib->xmab", self.quad_weights, S, S)
        return jnp.sum(jnp.prod(gram, axis=1), axis=(1, 2))

    def _class_quadratics(self, cores, grids):
        Gc = self.class_cores(cores)
        S = self.target_values(grids)

        def one(Gx, Wx, right):
            L0 = jnp.outer(self.op_left, self.op_left)
            Lf, _ = lax.scan(lambda L, x: (jnp.einsum("ac,iab,icd,i->bd", L, x[0], x[0], x[1]), None),
                             L0, (Gx, Wx))
            return jnp.einsum("ac,a,c->", Lf, right, right)

        qmm = vmap(one)(Gc, self.quad_weights, self.class_right)
        _, _, qmf = self._env_mf(Gc, S)
        qff = self._quad_ff(S)
        return qmm - 2.0 * self.class_tau * qmf + self.class_tau * qff

    def loss(self, theta, grids, scales):
        quads = self._class_quadratics(self.core_channels(theta, grids, scales), grids)
        return 0.5 * jnp.dot(self.class_omega, quads)

    def loss_components(self, theta, grids, scales):
        quads = self._class_quadratics(self.core_channels(theta, grids, scales), grids)
        interior = 0.5 * quads[0]
        boundary = 0.5 * jnp.mean(quads[1:1 + 2 * self.dx])
        initial = 0.5 * quads[self.initial_class]
        total = interior + self.cfg.boundary_weight * boundary + self.cfg.initial_weight * initial
        return jnp.array([total, interior, boundary, initial])

    # ---------------- loss and compressed gradient ----------------
    def loss_and_gradient(self, theta, grids, scales):
        """Loss and compressed gradient grad L = C^T rho_c.

        rho_c = sum_c omega_c rho_c^{(class)} is assembled slot-wise from the
        class environments (interior block chain, value chains on the traces),
        exactly as in the Gauss--Newton notebook; C^T is applied as a single
        vector--Jacobian product of the local core channels (C is never formed).
        """
        local, pull = jax.vjp(lambda th: self.local_channels(th, grids, scales), theta)
        cores = self.pad_channels(local)
        Gc = self.class_cores(cores)
        S = self.target_values(grids)
        W, om, tau = self.quad_weights, self.class_omega, self.class_tau
        T = self._transfers(Gc)
        Lmm, Rmm, qmm = self._env_mm(T)
        Lmf, Rmf, qmf = self._env_mf(Gc, S)
        qff = self._quad_ff(S)
        loss = 0.5 * jnp.dot(om, qmm - 2.0 * tau * qmf + tau * qff)

        cotangents = []
        for run in self.runs:
            sl = slice(run.start, run.stop)
            Ps = self.insertions(run)

            def rho_slot(G_s, W_s, L_s, R_s, Lf_s, Rf_s, S_s, Ps=Ps):
                OW = om[:, None] * W_s
                Wm = jnp.einsum("xpae,xbe->xpab", jnp.einsum("xac,xpce->xpae", L_s, G_s), R_s)
                Wf = jnp.einsum("xac,pc,xbc->xpab", Lf_s, S_s, Rf_s)
                Wr = (jnp.einsum("xp,xpab->pab", OW, Wm)
                      - jnp.einsum("xp,xpab->pab", OW * tau[:, None], Wf))
                return jnp.einsum("luab,pab->plu", Ps, Wr)       # rho_c block as (i_s, l, a_s)

            cotangents.append(vmap(rho_slot, in_axes=(1, 1, 1, 1, 1, 1, 0))(
                Gc[:, sl], W[:, sl], Lmm[:, sl], Rmm[:, sl], Lmf[:, sl], Rmf[:, sl], S[sl]))
        grad = pull(tuple(cotangents))[0]
        return loss, grad

    def autodiff_gradient(self, theta, grids, scales):
        """Reference: reverse-mode gradient of the factorized loss (diagnostics only)."""
        return jax.grad(lambda th: self.loss(th, grids, scales))(theta)

    def solver_used(self):
        return "adam"

    @property
    def N_axis(self):
        return max(self.N_slots)

    # ---------------- exact error and evaluation ----------------
    def relative_error(self, theta, scales):
        """||u_theta - u*|| / ||u*|| in L2(D) with a midpoint product rule."""
        n = self.cfg.n_err
        pts = (jnp.arange(n) + 0.5) / n
        grids = jnp.tile(pts[None, :], (self.d, 1))
        U = self.core_values(theta, grids, scales)
        T = self.target_values(grids)
        mm, _ = lax.scan(lambda s, A: (jnp.einsum("ac,iab,icd->bd", s, A, A) / n, None),
                         jnp.outer(self.lvec, self.lvec), U)
        mm = jnp.einsum("ac,a,c->", mm, self.rvec, self.rvec)
        mt, _ = lax.scan(lambda s, x: (jnp.einsum("ac,iab,ic->bc", s, x[0], x[1]) / n, None),
                         jnp.outer(self.lvec, self.ones_K), (U, T))
        mt = jnp.einsum("ac,a,c->", mt, self.rvec, self.ones_K)
        tt = jnp.sum(jnp.prod(jnp.einsum("mia,mib->mab", T, T) / n, axis=0))
        return jnp.sqrt(jnp.maximum(mm - 2.0 * mt + tt, 0.0) / tt)

    def model_points(self, theta, X, scales):
        U = self.core_values(theta, X.T, scales)
        st, _ = lax.scan(lambda s, A: (jnp.einsum("na,nab->nb", s, A), None),
                         jnp.tile(self.lvec[None, :], (X.shape[0], 1)), U)
        return st @ self.rvec

    def exact_points(self, X):
        return jnp.prod(self.target_factors(X.T), axis=0) @ self.C_target

    def model_slice(self, theta, xs, ys, fixed, scales, axes=(0, 1)):
        s, t = axes
        fixed = jnp.asarray(fixed)
        g_fix = self.core_values(theta, fixed[:, None], scales)[:, 0]
        g_s = self.core_values(theta, jnp.tile(xs[None, :], (self.d, 1)), scales)[s]
        g_t = self.core_values(theta, jnp.tile(ys[None, :], (self.d, 1)), scales)[t]
        lo, hi = min(s, t), max(s, t)
        left = self.lvec
        for m in range(lo):
            left = left @ g_fix[m]
        right = self.rvec
        for m in range(self.d - 1, hi, -1):
            right = g_fix[m] @ right
        mid = jnp.eye(self.c)
        for m in range(lo + 1, hi):
            mid = mid @ g_fix[m]
        g_lo, g_hi = (g_s, g_t) if s < t else (g_t, g_s)
        a = jnp.einsum("a,iab->ib", left, g_lo) @ mid
        b = jnp.einsum("jbc,c->jb", g_hi, right)
        out = a @ b.T
        return out if s < t else out.T

    def exact_slice(self, xs, ys, fixed, axes=(0, 1)):
        s, t = axes
        others = jnp.array([m for m in range(self.d) if m not in axes])
        T_full = self.target_factors(jnp.asarray(fixed)[:, None])
        w = jnp.prod(T_full[others, 0, :], axis=0) * self.C_target
        T_s = self.target_factors(jnp.tile(xs[None, :], (self.d, 1)))[s]
        T_t = self.target_factors(jnp.tile(ys[None, :], (self.d, 1)))[t]
        return (T_s * w[None, :]) @ T_t.T

    # ---------------- initialization and diagnostics ----------------
    def factor_scales(self, theta, n_probe=512):
        probe = (jnp.arange(n_probe) + 0.5) / n_probe
        grids = jnp.tile(probe[None, :], (self.d, 1))
        unit = jnp.ones((self.d,))
        U = self.core_values(theta, grids, unit)
        time_integral = (1.0 - jnp.exp(-2.0 * self.decay)) / (2.0 * self.decay)
        target_norm = jnp.sqrt(0.5 * self.dx * time_integral)
        if (self.cfg.tt_init_mode.lower() == "near_identity"
                and self.cfg.tt_global_scale):
            st, _ = lax.scan(lambda s, A: (jnp.einsum("ac,iab,icd->bd", s, A, A) / n_probe, None),
                             jnp.outer(self.lvec, self.lvec), U)
            raw_norm = jnp.sqrt(jnp.maximum(jnp.einsum("ac,a,c->", st, self.rvec, self.rvec), 1e-24))
            return unit.at[0].set(target_norm / raw_norm)
        k = jnp.asarray(self.k_slots, dtype=U.dtype)
        rms = jnp.sqrt(jnp.sum(U**2, axis=(1, 2, 3)) / (n_probe * k))
        paths = float(self.c) ** (self.d - 1)
        sigma = (target_norm / jnp.sqrt(paths)) ** (1.0 / self.d)
        return sigma / jnp.maximum(rms, 1e-12)

    def initialization_diagnostics(self, theta, scales, n_probe=256):
        probe = (jnp.arange(n_probe) + 0.5) / n_probe
        identity = jnp.eye(self.c).reshape(-1)
        devs = []
        for run, th in zip(self.runs, theta):
            raw = vmap(lambda t: vmap(lambda z: self.mlp(t, z, run.k))(probe))(th)
            devs.append(jnp.sqrt(jnp.mean((raw - identity[jnp.asarray(run.units)]) ** 2, axis=(1, 2))))
        dev = jnp.concatenate(devs)
        return dict(
            init_mode=self.cfg.tt_init_mode.lower(),
            global_scale=float(np.asarray(scales[0])) if self.cfg.tt_global_scale else None,
            mean_core_identity_deviation=float(np.asarray(jnp.mean(dev))),
            max_core_identity_deviation=float(np.asarray(jnp.max(dev))),
        )

    def dof(self):
        d, R = self.d, self.c
        gauge = (d - 1) * R * R
        return dict(P=self.P, P_axis=self.P_axis, gauge=int(gauge), P_eff_analytic=int(self.P - gauge))


def scale_mlp(model, theta_slot, z, scale, k):
    return scale * model.mlp(theta_slot, z, k)


# ============================================================
# Training in compiled chunks, with sparse error evaluation
# ============================================================
def adam_learning_rate(cfg, step):
    """eta_k = eta_0 (eta_end / eta_0)^(k / K), k = 0, ..., K - 1."""
    frac = jnp.minimum(step / max(cfg.iterations, 1), 1.0)
    return cfg.adam_lr * (cfg.adam_lr_final / cfg.adam_lr) ** frac


def make_trainer(model: Model):
    cfg = model.cfg
    b1, b2, eps = cfg.adam_beta1, cfg.adam_beta2, cfg.adam_eps

    def iteration(carry, _):
        theta, m, v, t, key, scales, grids = carry
        if cfg.resample_each_iteration:
            key, k_grid = random.split(key)
            grids = model.sample_grids(k_grid)
        loss, grad = model.loss_and_gradient(theta, grids, scales)
        finite = jnp.all(jnp.stack([jnp.all(jnp.isfinite(g)) for g in grad]))
        grad = tree_map(lambda g: jnp.where(finite, g, 0.0), grad)
        lr = adam_learning_rate(cfg, t)
        t = t + 1
        m = tree_map(lambda mm, g: b1 * mm + (1.0 - b1) * g, m, grad)
        v = tree_map(lambda vv, g: b2 * vv + (1.0 - b2) * g * g, v, grad)
        c1 = 1.0 / (1.0 - b1 ** t)
        c2 = 1.0 / (1.0 - b2 ** t)
        theta = tree_map(lambda th, mm, vv: th - lr * (mm * c1) / (jnp.sqrt(vv * c2) + eps), theta, m, v)
        return (theta, m, v, t, key, scales, grids), (loss, lr)

    @partial(jit, static_argnums=7)
    def train_chunk(theta, m, v, t, key, scales, grids, n_iter):
        (theta, m, v, t, key, _, _), hist = lax.scan(
            iteration, (theta, m, v, t, key, scales, grids), None, length=n_iter
        )
        return theta, m, v, t, key, hist

    return train_chunk


def _training_chunks(iterations, error_every):
    if iterations <= 0:
        raise ValueError("iterations must be positive.")
    if error_every <= 0:
        raise ValueError("error_every must be positive.")
    checkpoints = list(range(error_every, iterations + 1, error_every))
    if not checkpoints or checkpoints[-1] != iterations:
        checkpoints.append(iterations)
    chunk_sizes = np.diff(np.asarray([0] + checkpoints, dtype=int))
    return np.asarray(checkpoints, dtype=int), chunk_sizes


def run(cfg: Config, verbose=True, monitor_progress=False):
    model = Model(cfg)
    train_chunk = make_trainer(model)
    relative_error = jit(lambda theta, scales: model.relative_error(theta, scales))
    loss_components = jit(lambda theta, grids, scales: model.loss_components(theta, grids, scales))

    root_key = random.PRNGKey(cfg.seed)
    k_init, k_grid, train_key0 = random.split(root_key, 3)
    theta0 = model.init_theta(k_init)
    scales = model.factor_scales(theta0)
    init_diag = model.initialization_diagnostics(theta0, scales)
    grids = model.sample_grids(k_grid)
    checkpoints, chunk_sizes = _training_chunks(cfg.iterations, cfg.error_every)

    # Compilation is measured separately and is excluded from the wall-clock
    # history used in the convergence plots.
    m0 = tree_map(jnp.zeros_like, theta0)
    v0 = tree_map(jnp.zeros_like, theta0)
    t_adam0 = jnp.asarray(0, dtype=jnp.int32)
    loss_fn = jit(lambda theta_, grids_, scales_: model.loss(theta_, grids_, scales_))

    t0 = time.perf_counter()
    for n_chunk in sorted(set(int(n) for n in chunk_sizes)):
        train_chunk.lower(theta0, m0, v0, t_adam0, train_key0, scales, grids, n_chunk).compile()
    loss_fn.lower(theta0, grids, scales).compile()
    relative_error.lower(theta0, scales).compile()
    loss_components.lower(theta0, grids, scales).compile()
    t_compile = time.perf_counter() - t0

    err0 = relative_error(theta0, scales)
    jax.block_until_ready(err0)

    if monitor_progress:
        print(
            f"[training] iter {0:5d}/{cfg.iterations} | "
            f"rel L2 = {float(np.asarray(err0)):.6e}",
            flush=True,
        )

    theta, train_key = theta0, train_key0
    m_adam, v_adam, t_adam = m0, v0, t_adam0
    loss_parts, step_parts = [], []
    iteration_times = []
    error_iterations = [0]
    error_values = [float(np.asarray(err0))]
    error_times = [0.0]
    cumulative_iteration_time = 0.0

    for stop_iteration, n_chunk in zip(checkpoints, chunk_sizes):
        n_chunk = int(n_chunk)

        t0 = time.perf_counter()
        theta, m_adam, v_adam, t_adam, train_key, hist = train_chunk(
            theta, m_adam, v_adam, t_adam, train_key, scales, grids, n_chunk)
        jax.block_until_ready(theta)
        dt = time.perf_counter() - t0

        cumulative_iteration_time += dt
        iteration_times.extend([dt / n_chunk] * n_chunk)

        accepted_loss_chunk, step_chunk = [np.asarray(h) for h in hist]
        loss_parts.append(accepted_loss_chunk)
        step_parts.append(step_chunk)

        err = relative_error(theta, scales)
        jax.block_until_ready(err)
        err_value = float(np.asarray(err))
        error_iterations.append(int(stop_iteration))
        error_values.append(err_value)
        error_times.append(cumulative_iteration_time)

        if monitor_progress:
            loss_value = float(accepted_loss_chunk[-1])
            step_value = float(step_chunk[-1])
            print(
                f"[training] iter {int(stop_iteration):5d}/{cfg.iterations} | "
                f"rel L2 = {err_value:.6e} | "
                f"loss = {loss_value:.6e} | "
                f"lr = {step_value:.3e} | "
                f"train time = {cumulative_iteration_time:.2f} s",
                flush=True,
            )

    loss_h = np.concatenate(loss_parts)
    step_h = np.concatenate(step_parts)
    iteration_time_h = np.asarray(iteration_times, dtype=float)
    cumulative_iteration_time_h = np.cumsum(iteration_time_h)
    err_h = np.asarray(error_values, dtype=float)

    components = np.asarray(loss_components(theta, grids, scales), dtype=float)
    component_names = ("total", "interior", "boundary_average", "initial")
    component_record = dict(zip(component_names, map(float, components)))

    # loss[k] is the loss at the iterate before the k-th Adam update;
    # loss_after[k] is the loss after it.  step stores the learning rate.
    final_loss = float(np.asarray(loss_fn(theta, grids, scales)))
    history = dict(
        loss=loss_h,
        loss_after=np.append(loss_h[1:], final_loss),
        step=step_h,
        learning_rate=step_h,
        iteration_time=iteration_time_h,
        cumulative_iteration_time=cumulative_iteration_time_h,
        error_iterations=np.asarray(error_iterations, dtype=int),
        relative_l2=err_h,
        error_times=np.asarray(error_times, dtype=float),
    )

    t_run = float(cumulative_iteration_time)
    rec = dict(
        **asdict(cfg),
        d=cfg.d,
        **model.dof(),
        active_rank=model.c,
        active_width=cfg.active_width,
        active_hidden=cfg.active_hidden,
        parameter_block="TT core network",
        network_shape=tuple(int(s) for s in model.sizes),
        P_slots=tuple(int(p) for p in model.P_slots),
        network_shapes=tuple(tuple(int(s) for s in model.network_sizes(r)) for r in model.runs),
        tt_ranks=(tuple(int(r) for r in model.ranks) if model.ranks is not None else None),
        local_channels="Lambda_s={0,ell_s}, ell_s=2 (space), 1 (time)",
        block_core_size=model.q,
        residual_classes=model.n_classes,
        augmented_grid_points_per_axis=model.n_grid,
        solver_used=model.solver_used(),
        factorization="none",
        optimizer="Adam (compressed gradient C^T rho_c)",
        sampling_policy=("resample_each_iteration" if cfg.resample_each_iteration else "fixed"),
        initialization=init_diag,
        N_axis=model.N_axis,
        N_slots=tuple(int(k) for k in model.N_slots),
        N_eff=model.N_eff,
        err_initial=float(err_h[0]),
        err_final=float(err_h[-1]),
        err_best=float(err_h.min()),
        loss_final=final_loss,
        loss_components_final=component_record,
        loss_total_components_final=component_record["total"],
        loss_pde_final=component_record["interior"],
        loss_boundary_final=component_record["boundary_average"],
        loss_initial_final=component_record["initial"],
        loss_boundary_weighted_final=float(cfg.boundary_weight * component_record["boundary_average"]),
        loss_initial_weighted_final=float(cfg.initial_weight * component_record["initial"]),
        error_evaluations=int(err_h.size),
        t_compile=float(t_compile),
        t_run=t_run,
        t_per_iter=t_run / cfg.iterations,
    )

    if verbose:
        parameter_name = "P_core"
        print(
            f"[{cfg.format}/{model.solver_used()}] d={cfg.spatial_dim}+1 rank={model.c} "
            f"width={cfg.active_width} seed={cfg.seed} | "
            f"{parameter_name}={model.P_axis}, P_total={model.P}, N_eff={model.N_eff} "
            f"| sampling={rec['sampling_policy']} | err={rec['err_final']:.3e} "
            f"| {1e3 * rec['t_per_iter']:.2f} ms/it, {rec['t_run']:.1f} s"
        )
        if True:
            print(
                "  TT initialization: "
                f"mode={init_diag['init_mode']}, "
                f"global_scale={init_diag['global_scale']}, "
                f"mean ||G-I||_RMS={init_diag['mean_core_identity_deviation']:.3e}, "
                f"max={init_diag['max_core_identity_deviation']:.3e}"
            )
        print(
            "  final loss components: "
            f"PDE={component_record['interior']:.3e}, "
            f"boundary(avg)={component_record['boundary_average']:.3e} "
            f"(lambda_b={cfg.boundary_weight:g}, weighted="
            f"{cfg.boundary_weight * component_record['boundary_average']:.3e}), "
            f"initial={component_record['initial']:.3e} "
            f"(lambda_0={cfg.initial_weight:g}, weighted="
            f"{cfg.initial_weight * component_record['initial']:.3e}), "
            f"weighted total={component_record['total']:.3e}"
        )
    return model, theta, scales, rec, history


def width_for_budget(target_P, cfg: Config, w_max=256):
    best = None
    for w in range(2, w_max + 1):
        P = Model(replace(cfg, tt_width=w)).P
        if best is None or abs(P - target_P) < abs(best[1] - target_P):
            best = (w, P)
        if P >= target_P:
            break
    return best


def print_configured_parameter_counts(cfg: Config):
    model = Model(cfg)
    shapes = sorted({model.network_sizes(r) for r in model.runs}, key=lambda s: -s[-1])
    print("Configured TT architecture")
    print(
        f"TT core networks {' / '.join(str(s) for s in shapes)}, "
        f"P_core={' / '.join(str(p) for p in sorted(set(model.P_slots), reverse=True))}, "
        f"number of coordinate networks={cfg.d}, P_TT={model.P}, N_eff={model.N_eff}"
    )


## Hyperparameters

In [ ]:
# ------------------------------------------------------------
# User configuration: one TT-PINN training run
# ------------------------------------------------------------

# PDE and soft constraints.
SPATIAL_DIM = 10
DIFFUSIVITY = 0.25
SOFT_BOUNDARY_WEIGHT = 10.0
SOFT_INITIAL_WEIGHT = 5.0
BOUNDARY_TARGET = "exact"   # "exact" (consistent manufactured problem) or "zero"

# Factorized collocation and training.
#N_PER_AXIS = 32   # interior points per axis; 0 and 1 are appended internally
N_PER_AXIS = 32
ITERATIONS = 50000
ERROR_EVERY = 100
PRINT_TRAINING_PROGRESS = True  # print error/loss every ERROR_EVERY iterations
SEED = 40
N_ERR = 256

# False: one fixed set of factor grids per run.
# True: resample the interior factor-grid points at every Adam iteration.
RESAMPLE_EACH_ITERATION = False

# TT setup: ranks (1, r, ..., r, 1); core networks 1 -> w -> ... -> w -> r_(s-1) r_s.
TT_RANK = 2
TT_WIDTH = 128
#TT_HIDDEN = 3
TT_HIDDEN = 4

# TT initialization.
# "near_identity": G_m(z_m) = I + eps H_m(z_m); hidden layers remain fully active.
# "random"       : recover the original fan-in random initialization.
TT_INIT_MODE = "near_identity"
TT_IDENTITY_EPS = 0.5
# If True, keep the cores near identity and use one fixed global amplitude,
# calibrated by a factorized L2 transfer contraction at initialization.
TT_GLOBAL_SCALE = True

# Adam with exponential learning-rate decay eta_k = eta_0 (eta_end / eta_0)^(k / K).
ADAM_LR = 1e-3
ADAM_LR_FINAL = 1e-5      # set = ADAM_LR for a constant learning rate
ADAM_BETA1, ADAM_BETA2, ADAM_EPS = 0.9, 0.999, 1e-8

# ------------------------------------------------------------
# Random-seed ablation
# ------------------------------------------------------------
RUN_ABLATION = True
ABLATION_SEEDS = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
ABLATION_ITERATIONS = 50000
ABLATION_ERROR_EVERY = 100
ABLATION_TT_SOLVER = "adam"   # recorded in the exports
ABLATION_RESULT_DIR = "."
CLEAR_JAX_CACHES_BETWEEN_RUNS = True

BASE = Config(
    spatial_dim=SPATIAL_DIM,
    diffusivity=DIFFUSIVITY,
    boundary_weight=SOFT_BOUNDARY_WEIGHT,
    initial_weight=SOFT_INITIAL_WEIGHT,
    boundary_target=BOUNDARY_TARGET,
    n_per_axis=N_PER_AXIS,
    iterations=ITERATIONS,
    error_every=ERROR_EVERY,
    seed=SEED,
    n_err=N_ERR,
    resample_each_iteration=RESAMPLE_EACH_ITERATION,
    tt_rank=TT_RANK,
    tt_width=TT_WIDTH,
    tt_hidden=TT_HIDDEN,
    tt_init_mode=TT_INIT_MODE,
    tt_identity_epsilon=TT_IDENTITY_EPS,
    tt_global_scale=TT_GLOBAL_SCALE,
    adam_lr=ADAM_LR,
    adam_lr_final=ADAM_LR_FINAL,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    adam_eps=ADAM_EPS,
)

print_configured_parameter_counts(BASE)

model = Model(BASE)
per_block_name = "P_core"

print("\nActive single-run configuration")
print(f"TT-PINN + Adam  dimension={BASE.spatial_dim}+1  diffusivity={BASE.diffusivity}")
print(
    f"soft weights: lambda_b={BASE.boundary_weight} (averaged over {2 * BASE.spatial_dim} faces), "
    f"lambda_0={BASE.initial_weight}  boundary target={BASE.boundary_target}"
)
print(f"rank={model.c}  width={BASE.active_width}  hidden={BASE.active_hidden}")
print(
    f"TT init={BASE.tt_init_mode}, eps={BASE.tt_identity_epsilon}, "
    f"global amplitude scaling={BASE.tt_global_scale}"
)
print(f"TT ranks (r_0, r_1, ..., r_d) = (1, {model.c}, ..., {model.c}, 1)")
print("slot runs:")
for _run in model.runs:
    _slots = f"s = {_run.start + 1}" if _run.length == 1 else f"s = {_run.start + 1}..{_run.stop}"
    print(
        f"  {_slots:<10s} {'time' if _run.is_time else 'space':<6s} k_s = {_run.k:>3d}  L_s = {_run.L}  "
        f"N_s = {_run.N_slot:>6d}  {per_block_name} = {_run.P_slot:>6d}  net = {model.network_sizes(_run)}"
    )
print(f"P_TT={model.P}")
print(
    f"Adam: {BASE.iterations} iterations, lr {BASE.adam_lr:g} -> {BASE.adam_lr_final:g} (exponential), "
    f"betas=({BASE.adam_beta1}, {BASE.adam_beta2}), eps={BASE.adam_eps:g}; error every {BASE.error_every}"
)
print(
    f"factor grids: {BASE.n_per_axis} interior points + endpoints per axis "
    f"({model.n_grid} stored points per axis); tensor grids never formed"
)
print(f"compressed-gradient dimension N_eff^ch = 2 sum_s n_s r_(s-1) r_s = {model.N_eff}")
assert model.N_eff == heat_N_eff(BASE.d, model.n_grid, model.c)


## Training run

Trains one seed (`SEED`) and reports the relative $L^2$ error every `ERROR_EVERY` iterations.
`history["loss"]` is the loss before each Adam update. The last lines compare the compressed
gradient with reverse-mode automatic differentiation.

In [ ]:
model, theta, scales, record, history = run(
    BASE,
    monitor_progress=PRINT_TRAINING_PROGRESS,
)

loss_h = history["loss"]
step_h = history["step"]
err_iter_h = history["error_iterations"]
err_h = history["relative_l2"]
err_time_h = history["error_times"]
components = record["loss_components_final"]

print()
print(f"final relative L2 error: {record['err_final']:.4e}")
print(f"best  relative L2 error: {record['err_best']:.4e}")
print(f"time per iteration:      {1e3 * record['t_per_iter']:.0f} ms")
print(f"timed iteration total:   {record['t_run']:.3f} s")
print(f"error evaluations:       {record['error_evaluations']}")
print(f"solver actually used:    {record['solver_used']}")
if BASE.format.lower() == "tt":
    _init = record["initialization"]
    print(
        f"TT initialization:       {_init['init_mode']} "
        f"(eps={BASE.tt_identity_epsilon}, global scale={_init['global_scale']:.6e})"
    )
    print(
        "initial core deviation: "
        f"mean={_init['mean_core_identity_deviation']:.3e}, "
        f"max={_init['max_core_identity_deviation']:.3e}"
    )
print(f"final PDE loss:          {components['interior']:.4e}")
print(
    f"final boundary loss:     {components['boundary_average']:.4e}  "
    f"(unweighted; lambda_b={BASE.boundary_weight:g}, weighted="
    f"{BASE.boundary_weight * components['boundary_average']:.4e})"
)
print(
    f"final initial loss:      {components['initial']:.4e}  "
    f"(unweighted; lambda_0={BASE.initial_weight:g}, weighted="
    f"{BASE.initial_weight * components['initial']:.4e})"
)

# Compressed gradient C^T rho_c versus reverse-mode differentiation of the loss.
_grids = model.sample_grids(random.PRNGKey(7))
_, _g = model.loss_and_gradient(theta, _grids, scales)
_ga = model.autodiff_gradient(theta, _grids, scales)
_flat = lambda tree: jnp.concatenate([x.reshape(-1) for x in tree])
print(
    "compressed gradient vs autodiff (relative max error): "
    f"{float(jnp.max(jnp.abs(_flat(_g) - _flat(_ga))) / jnp.max(jnp.abs(_flat(_ga)))):.3e}"
)


## Training diagnostics

Loss, relative $L^2$ error and learning rate of the training run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
fig.suptitle(
    f"{BASE.format.upper()} heat equation, d={BASE.spatial_dim}+1, "
    f"rank={model.c}, solver={model.solver_used()}, seed={BASE.seed}"
)
iteration_axis = np.arange(1, BASE.iterations + 1)

axes[0].plot(iteration_axis, loss_h, label="loss")
axes[0].set_yscale("log")
axes[0].set_title("weighted PINN loss")
axes[0].set_xlabel("iteration")
axes[0].legend()

axes[1].plot(err_iter_h, err_h, marker="o", ms=3)
axes[1].set_yscale("log")
axes[1].set_title(f"relative L2 error (every {BASE.error_every} iterations)")
axes[1].set_xlabel("iteration")

axes[2].plot(iteration_axis, step_h)
axes[2].set_yscale("log")
axes[2].set_title("Adam learning rate")
axes[2].set_xlabel("iteration")
plt.tight_layout()
plt.show()


## Solution plots

A time cut at $x_m=1/8$, and the exact and predicted $(x_1,x_2)$ slices at $t=0.5$. No
high-dimensional grid is formed.

In [ ]:
# ------------------------------------------------------------
# Time cut and two-dimensional spatial slice. No high-dimensional grid is formed.
# ------------------------------------------------------------
N_LINE = 400
N_SLICE = 200

# x_m=1/8 makes sin(2*pi*x_m)=1, so the time cut is nontrivial.
fixed = jnp.full((BASE.d,), 0.125)
fixed = fixed.at[model.time_slot].set(0.5)

# Time cut.
time_values = jnp.linspace(0.0, 1.0, N_LINE)
line_X = jnp.tile(fixed[None, :], (N_LINE, 1))
line_X = line_X.at[:, model.time_slot].set(time_values)
line_pred = model.model_points(theta, line_X, scales)
line_exact = model.exact_points(line_X)

# x_1-x_2 slice at t=0.5; all other spatial coordinates are fixed at 1/8.
xs = jnp.linspace(0.0, 1.0, N_SLICE)
ys = jnp.linspace(0.0, 1.0, N_SLICE)
pred_slice = model.model_slice(theta, xs, ys, fixed, scales, axes=(0, 1))
exact_slice = model.exact_slice(xs, ys, fixed, axes=(0, 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(np.asarray(time_values), np.asarray(line_exact), label="exact")
axes[0].plot(np.asarray(time_values), np.asarray(line_pred), "--", label="model")
axes[0].set_title(r"Time cut at $x_m=1/8$")
axes[0].set_xlabel("t")
axes[0].legend()

im1 = axes[1].imshow(
    np.asarray(exact_slice), origin="lower", extent=(0, 1, 0, 1), aspect="auto"
)
axes[1].set_title(r"Exact $x_1$--$x_2$ slice at $t=0.5$")
axes[1].set_xlabel(r"$x_2$")
axes[1].set_ylabel(r"$x_1$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(
    np.asarray(pred_slice), origin="lower", extent=(0, 1, 0, 1), aspect="auto"
)
axes[2].set_title(r"Model $x_1$--$x_2$ slice at $t=0.5$")
axes[2].set_xlabel(r"$x_2$")
axes[2].set_ylabel(r"$x_1$")
fig.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()


## Seed ablation

Only the random seed varies; problem, collocation, weights, architecture, initialization and optimizer
are fixed. The first cell trains the ten seeds in `ABLATION_SEEDS`; the next cells plot and export.

In [ ]:
# ------------------------------------------------------------
# Run the fixed-configuration TT random-seed ablation.
# This cell trains only; plotting and reporting are in the next cell.
# ------------------------------------------------------------
import gc

if RUN_ABLATION:
    if not ABLATION_SEEDS:
        raise ValueError("ABLATION_SEEDS must contain at least one seed.")

    seeds = [int(seed) for seed in ABLATION_SEEDS]
    if len(seeds) != 10:
        raise ValueError(
            f"This ablation is configured for exactly 10 seeds; received {len(seeds)}."
        )
    if len(set(seeds)) != len(seeds):
        raise ValueError("ABLATION_SEEDS must not contain duplicates.")

    ablation_results = {}
    ablation_records = []

    print(f"\n{'=' * 72}\nTT + Adam ablation over 10 random seeds\n{'=' * 72}")

    for run_index, seed in enumerate(seeds, start=1):
        print(f"\nTT seed {seed} ({run_index}/{len(seeds)})")
        cfg = replace(
            BASE,
            format="tt",
            iterations=ABLATION_ITERATIONS,
            error_every=ABLATION_ERROR_EVERY,
            seed=seed,
        )

        ab_model, ab_theta, ab_scales, rec, hist = run(cfg, verbose=True)
        ablation_results[seed] = {"record": rec, "history": hist}
        ablation_records.append(rec)

        del ab_model, ab_theta, ab_scales
        gc.collect()
        if CLEAR_JAX_CACHES_BETWEEN_RUNS:
            jax.clear_caches()

    completed_runs = len(ablation_results)
    if completed_runs != len(seeds):
        raise RuntimeError(
            f"Ablation finished with {completed_runs}/{len(seeds)} completed TT runs."
        )

    print(f"\nTT ablation completed: {completed_runs} runs.")
    print("Run the next cell to create the mean/IQR convergence figure and exports.")
else:
    ablation_results = {}
    ablation_records = []
    print("Ablation skipped because RUN_ABLATION=False.")


### Figures and export

Solid curves show the median over the ten seeds, shaded bands the interquartile range (25th–75th
percentiles). The wall-clock panel uses the median cumulative iteration time at each error
checkpoint. The cell also exports the per-seed table and full histories.

In [ ]:
# ------------------------------------------------------------
# Plot, summarize, and export the completed TT random-seed ablation.
# No model training is performed in this cell.
#
# Central curve : median across seeds.
# Shaded band   : interquartile range (IQR), Q1--Q3.
# ------------------------------------------------------------

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def _json_history(history):
    return {
        name: np.asarray(values).tolist()
        for name, values in history.items()
    }


def _stack_equal(arrays, name):
    arrays = [np.asarray(a, dtype=float) for a in arrays]
    shapes = {a.shape for a in arrays}

    if len(shapes) != 1:
        raise RuntimeError(
            f"Inconsistent {name} history shapes across seeds: "
            f"{sorted(shapes)}"
        )

    return np.stack(arrays, axis=0)


def _median_iqr(stack):
    """
    Compute statistics across random seeds.

    Returns
    -------
    median : pointwise median (50th percentile)
    q25    : pointwise first quartile (25th percentile)
    q75    : pointwise third quartile (75th percentile)

    The interval [q25, q75] contains the middle 50% of runs.
    """
    median = np.median(stack, axis=0)
    q25, q75 = np.percentile(
        stack,
        [25.0, 75.0],
        axis=0,
    )
    return median, q25, q75


# ------------------------------------------------------------
# Check that the ablation has been completed.
# ------------------------------------------------------------

if "ablation_results" not in globals():
    raise RuntimeError(
        "Run the TT ablation-training cell above before plotting."
    )

seeds = [int(seed) for seed in ABLATION_SEEDS]

missing = [
    seed
    for seed in seeds
    if seed not in ablation_results
]

if missing:
    raise RuntimeError(
        "The TT ablation is incomplete. "
        "Run the training cell again. "
        f"Missing seeds: {missing}"
    )

if len(seeds) != 10:
    raise RuntimeError(
        f"Expected 10 TT seeds, found {len(seeds)}."
    )


result_dir = Path(ABLATION_RESULT_DIR)
result_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Stack histories from all ten random seeds.
# ------------------------------------------------------------

histories = [
    ablation_results[seed]["history"]
    for seed in seeds
]

loss_stack = _stack_equal(
    [h["loss"] for h in histories],
    "loss",
)

error_stack = _stack_equal(
    [h["relative_l2"] for h in histories],
    "relative L2",
)

time_stack = _stack_equal(
    [h["error_times"] for h in histories],
    "error time",
)


# ------------------------------------------------------------
# Verify common error-evaluation checkpoints.
# ------------------------------------------------------------

error_iterations_all = [
    np.asarray(
        h["error_iterations"],
        dtype=int,
    )
    for h in histories
]

reference_error_iterations = error_iterations_all[0]

if not all(
    np.array_equal(
        it,
        reference_error_iterations,
    )
    for it in error_iterations_all[1:]
):
    raise RuntimeError(
        "Relative-error checkpoint iterations differ across seeds."
    )


loss_iterations = np.arange(
    1,
    loss_stack.shape[1] + 1,
)

error_iterations = reference_error_iterations


# ------------------------------------------------------------
# Median and interquartile range across seeds.
#
# Q1 = 25th percentile.
# Q3 = 75th percentile.
#
# [Q1,Q3] contains the middle 50% of random-seed runs.
# ------------------------------------------------------------

loss_median, loss_q25, loss_q75 = _median_iqr(
    loss_stack
)

error_median, error_q25, error_q75 = _median_iqr(
    error_stack
)


# ------------------------------------------------------------
# Wall-clock times differ slightly between seeds.
# Use the median cumulative time at each common error checkpoint.
# ------------------------------------------------------------

time_median = np.median(
    time_stack,
    axis=0,
)

time_q25, time_q75 = np.percentile(
    time_stack,
    [25.0, 75.0],
    axis=0,
)


# ------------------------------------------------------------
# Publication-style 1 x 3 figure.
#
# Solid line : median.
# Shaded band: IQR = Q1--Q3.
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


fig, axes = plt.subplots(
    1,
    3,
    figsize=(10.6, 3.45),
    constrained_layout=True,
)


# ------------------------------------------------------------
# (1) Loss vs GN iterations.
# ------------------------------------------------------------

axes[0].plot(
    loss_iterations,
    loss_median,
    linewidth=2.0,
    label="median",
)

axes[0].fill_between(
    loss_iterations,
    loss_q25,
    loss_q75,
    alpha=0.20,
    label="IQR",
)

axes[0].set_title(
    "(a)"
)

axes[0].set_xlabel(
    "iteration"
)

axes[0].set_ylabel(
    "Loss"
)

axes[0].set_yscale(
    "log"
)


# ------------------------------------------------------------
# (2) Relative L2 error vs GN iterations.
# ------------------------------------------------------------

axes[1].plot(
    error_iterations,
    error_median,
    linewidth=2.0,
    label="median",
)

axes[1].fill_between(
    error_iterations,
    error_q25,
    error_q75,
    alpha=0.20,
    label="IQR",
)

axes[1].set_title(
    r"(b)"
)

axes[1].set_xlabel(
    "iteration"
)

axes[1].set_ylabel(
    r"Relative $L^2$ error"
)

axes[1].set_yscale(
    "log"
)


# ------------------------------------------------------------
# (3) Relative L2 error vs wall-clock time.
#
# We use the median cumulative time as the horizontal coordinate.
# ------------------------------------------------------------

axes[2].plot(
    time_median,
    error_median,
    linewidth=2.0,
    label="median",
)

axes[2].fill_between(
    time_median,
    error_q25,
    error_q75,
    alpha=0.20,
    label="IQR",
)

axes[2].set_title(
    r"(c)"
)

axes[2].set_xlabel(
    "Wall clock time [s]"
)

axes[2].set_ylabel(
    r"Relative $L^2$ error"
)

axes[2].set_yscale(
    "log"
)


# ------------------------------------------------------------
# Common formatting.
# ------------------------------------------------------------

for ax in axes:

    ax.grid(
        True,
        which="major",
        linewidth=0.4,
        alpha=0.20,
    )

    ax.grid(
        True,
        which="minor",
        linewidth=0.25,
        alpha=0.10,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.6)

    ax.tick_params(
        which="both",
        direction="out",
    )

    ax.legend()


fig.suptitle(
    rf"Tensor train + Adam: median over 10 random seeds with IQR bands, "
    rf"$\lambda_b={SOFT_BOUNDARY_WEIGHT:g}$, "
    rf"$\lambda_0={SOFT_INITIAL_WEIGHT:g}$",
    fontsize=12,
)


# ------------------------------------------------------------
# Export figure.
# ------------------------------------------------------------

figure_path = (
    result_dir
    / "tt_heat_adam_10seed_median_iqr.png"
)

figure_pdf_path = (
    result_dir
    / "tt_heat_adam_10seed_median_iqr.pdf"
)

fig.savefig(
    figure_path,
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    figure_pdf_path,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# One row per TT run.
# ------------------------------------------------------------

run_table = pd.DataFrame([
    ablation_results[seed]["record"]
    for seed in seeds
])

run_table["format"] = (
    run_table["format"].str.upper()
)

run_table["ms_per_iteration"] = (
    1e3 * run_table["t_per_iter"]
)


run_table = run_table.rename(
    columns={
        "active_rank": "model_rank",
        "active_width": "width",
        "active_hidden": "hidden_layers",
        "P_axis": "parameters_per_core",
        "P": "total_parameters",
        "loss_final": "final_loss",
        "err_final": "final_rel_L2",
        "err_best": "best_rel_L2",
        "loss_pde_final": "final_PDE_loss",
        "loss_boundary_final": "final_boundary_loss",
        "loss_initial_final": "final_initial_loss",
        "loss_boundary_weighted_final":
            "final_boundary_weighted_loss",
        "loss_initial_weighted_final":
            "final_initial_weighted_loss",
    }
)


run_columns = [
    "format",
    "seed",
    "solver_used",
    "network_shape",
    "model_rank",
    "width",
    "hidden_layers",
    "parameters_per_core",
    "total_parameters",
    "N_eff",
    "tt_init_mode",
    "tt_identity_epsilon",
    "tt_global_scale",
    "boundary_weight",
    "initial_weight",
    "boundary_target",
    "iterations",
    "error_every",
    "final_loss",
    "final_PDE_loss",
    "final_boundary_loss",
    "final_initial_loss",
    "final_boundary_weighted_loss",
    "final_initial_weighted_loss",
    "final_rel_L2",
    "best_rel_L2",
    "t_run",
    "ms_per_iteration",
]


run_table = (
    run_table[run_columns]
    .sort_values("seed")
    .reset_index(drop=True)
)


display(
    run_table.style.format({
        "final_loss": "{:.4e}",
        "final_PDE_loss": "{:.4e}",
        "final_boundary_loss": "{:.4e}",
        "final_initial_loss": "{:.4e}",
        "final_boundary_weighted_loss": "{:.4e}",
        "final_initial_weighted_loss": "{:.4e}",
        "final_rel_L2": "{:.4e}",
        "best_rel_L2": "{:.4e}",
        "t_run": "{:.3f}",
        "ms_per_iteration": "{:.2f}",
    })
)


# ------------------------------------------------------------
# Scalar robustness summary over the ten seeds.
#
# We report:
#     median
#     Q1 = 25th percentile
#     Q3 = 75th percentile
#     IQR = Q3 - Q1
# ------------------------------------------------------------

final_error_q25 = (
    run_table["final_rel_L2"].quantile(0.25)
)

final_error_q75 = (
    run_table["final_rel_L2"].quantile(0.75)
)

final_loss_q25 = (
    run_table["final_loss"].quantile(0.25)
)

final_loss_q75 = (
    run_table["final_loss"].quantile(0.75)
)

time_per_it_q25 = (
    run_table["ms_per_iteration"].quantile(0.25)
)

time_per_it_q75 = (
    run_table["ms_per_iteration"].quantile(0.75)
)


aggregate_table = pd.DataFrame([{

    "seeds":
        len(seeds),

    "final_rel_L2_median":
        run_table["final_rel_L2"].median(),

    "final_rel_L2_q25":
        final_error_q25,

    "final_rel_L2_q75":
        final_error_q75,

    "final_rel_L2_IQR":
        final_error_q75 - final_error_q25,

    "best_rel_L2_median":
        run_table["best_rel_L2"].median(),

    "final_loss_median":
        run_table["final_loss"].median(),

    "final_loss_q25":
        final_loss_q25,

    "final_loss_q75":
        final_loss_q75,

    "final_loss_IQR":
        final_loss_q75 - final_loss_q25,

    "ms_per_iteration_median":
        run_table["ms_per_iteration"].median(),

    "ms_per_iteration_q25":
        time_per_it_q25,

    "ms_per_iteration_q75":
        time_per_it_q75,

    "ms_per_iteration_IQR":
        time_per_it_q75 - time_per_it_q25,
}])


display(
    aggregate_table.style.format({

        "final_rel_L2_median": "{:.4e}",
        "final_rel_L2_q25": "{:.4e}",
        "final_rel_L2_q75": "{:.4e}",
        "final_rel_L2_IQR": "{:.4e}",

        "best_rel_L2_median": "{:.4e}",

        "final_loss_median": "{:.4e}",
        "final_loss_q25": "{:.4e}",
        "final_loss_q75": "{:.4e}",
        "final_loss_IQR": "{:.4e}",

        "ms_per_iteration_median": "{:.2f}",
        "ms_per_iteration_q25": "{:.2f}",
        "ms_per_iteration_q75": "{:.2f}",
        "ms_per_iteration_IQR": "{:.2f}",
    })
)


# ------------------------------------------------------------
# Aggregate histories used in the figure.
# ------------------------------------------------------------

loss_history_table = pd.DataFrame({

    "iteration":
        loss_iterations,

    "loss_median":
        loss_median,

    "loss_q25":
        loss_q25,

    "loss_q75":
        loss_q75,

    "loss_IQR":
        loss_q75 - loss_q25,
})


error_history_table = pd.DataFrame({

    "iteration":
        error_iterations,

    "median_wall_time_s":
        time_median,

    "wall_time_q25_s":
        time_q25,

    "wall_time_q75_s":
        time_q75,

    "relative_l2_median":
        error_median,

    "relative_l2_q25":
        error_q25,

    "relative_l2_q75":
        error_q75,

    "relative_l2_IQR":
        error_q75 - error_q25,
})


# ------------------------------------------------------------
# Export CSV files.
# ------------------------------------------------------------

run_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_runs.csv"
)

aggregate_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_summary_median_iqr.csv"
)

loss_history_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_loss_history_median_iqr.csv"
)

error_history_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_error_history_median_iqr.csv"
)

json_path = (
    result_dir
    / "tt_heat_adam_10seed_histories_median_iqr.json"
)


run_table.to_csv(
    run_csv_path,
    index=False,
)

aggregate_table.to_csv(
    aggregate_csv_path,
    index=False,
)

loss_history_table.to_csv(
    loss_history_csv_path,
    index=False,
)

error_history_table.to_csv(
    error_history_csv_path,
    index=False,
)


# ------------------------------------------------------------
# Complete JSON export.
# ------------------------------------------------------------

payload = {

    "settings": {

        "problem":
            "10+1 dimensional heat equation with "
            "soft spatial-boundary and initial residuals",

        "format":
            "tt",

        "spatial_dim":
            int(SPATIAL_DIM),

        "diffusivity":
            float(DIFFUSIVITY),

        "boundary_weight":
            float(SOFT_BOUNDARY_WEIGHT),

        "initial_weight":
            float(SOFT_INITIAL_WEIGHT),

        "boundary_target":
            BOUNDARY_TARGET,

        "seeds":
            seeds,

        "iterations":
            int(ABLATION_ITERATIONS),

        "error_every":
            int(ABLATION_ERROR_EVERY),

        "tt": {

            "rank":
                int(TT_RANK),

            "width":
                int(TT_WIDTH),

            "hidden":
                int(TT_HIDDEN),

            "solver":
                ABLATION_TT_SOLVER,

            "init_mode":
                TT_INIT_MODE,

            "identity_epsilon":
                float(TT_IDENTITY_EPS),

            "global_scale":
                bool(TT_GLOBAL_SCALE),
        },
    },


    "aggregate_history": {

        "statistics":
            "median with interquartile range "
            "(25th--75th percentiles)",

        "loss_iteration":
            loss_iterations.tolist(),

        "loss_median":
            loss_median.tolist(),

        "loss_q25":
            loss_q25.tolist(),

        "loss_q75":
            loss_q75.tolist(),

        "loss_IQR":
            (loss_q75 - loss_q25).tolist(),

        "error_iteration":
            error_iterations.tolist(),

        "median_wall_time_s":
            time_median.tolist(),

        "wall_time_q25_s":
            time_q25.tolist(),

        "wall_time_q75_s":
            time_q75.tolist(),

        "relative_l2_median":
            error_median.tolist(),

        "relative_l2_q25":
            error_q25.tolist(),

        "relative_l2_q75":
            error_q75.tolist(),

        "relative_l2_IQR":
            (error_q75 - error_q25).tolist(),
    },


    "runs": {

        str(seed): {

            "record":
                ablation_results[seed]["record"],

            "history":
                _json_history(
                    ablation_results[seed]["history"]
                ),
        }

        for seed in seeds
    },
}


with open(
    json_path,
    "w",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Report saved files.
# ------------------------------------------------------------

print(
    f"saved PNG figure:              {figure_path}"
)

print(
    f"saved PDF figure:              {figure_pdf_path}"
)

print(
    f"saved per-run table:           {run_csv_path}"
)

print(
    f"saved scalar summary:          {aggregate_csv_path}"
)

print(
    f"saved median/IQR loss history: {loss_history_csv_path}"
)

print(
    f"saved median/IQR error history:{error_history_csv_path}"
)

print(
    f"saved all TT histories:        {json_path}"
)

The last cell redraws the same figure in the minimal style used in the manuscript (no titles or
legends).

In [ ]:
# ------------------------------------------------------------
# Plot, summarize, and export the completed TT random-seed ablation.
# No model training is performed in this cell.
#
# Central curve : median across seeds.
# Shaded band   : interquartile range (IQR), Q1--Q3.
# ------------------------------------------------------------

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def _json_history(history):
    return {
        name: np.asarray(values).tolist()
        for name, values in history.items()
    }


def _stack_equal(arrays, name):
    arrays = [np.asarray(a, dtype=float) for a in arrays]
    shapes = {a.shape for a in arrays}

    if len(shapes) != 1:
        raise RuntimeError(
            f"Inconsistent {name} history shapes across seeds: "
            f"{sorted(shapes)}"
        )

    return np.stack(arrays, axis=0)


def _median_iqr(stack):
    """
    Compute statistics across random seeds.

    Returns
    -------
    median : pointwise median (50th percentile)
    q25    : pointwise first quartile (25th percentile)
    q75    : pointwise third quartile (75th percentile)

    The interval [q25, q75] contains the middle 50% of runs.
    """
    median = np.median(stack, axis=0)
    q25, q75 = np.percentile(
        stack,
        [25.0, 75.0],
        axis=0,
    )
    return median, q25, q75


# ------------------------------------------------------------
# Check that the ablation has been completed.
# ------------------------------------------------------------

if "ablation_results" not in globals():
    raise RuntimeError(
        "Run the TT ablation-training cell above before plotting."
    )

seeds = [int(seed) for seed in ABLATION_SEEDS]

missing = [
    seed
    for seed in seeds
    if seed not in ablation_results
]

if missing:
    raise RuntimeError(
        "The TT ablation is incomplete. "
        "Run the training cell again. "
        f"Missing seeds: {missing}"
    )

if len(seeds) != 10:
    raise RuntimeError(
        f"Expected 10 TT seeds, found {len(seeds)}."
    )


result_dir = Path(ABLATION_RESULT_DIR)
result_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Stack histories from all ten random seeds.
# ------------------------------------------------------------

histories = [
    ablation_results[seed]["history"]
    for seed in seeds
]

loss_stack = _stack_equal(
    [h["loss"] for h in histories],
    "loss",
)

error_stack = _stack_equal(
    [h["relative_l2"] for h in histories],
    "relative L2",
)

time_stack = _stack_equal(
    [h["error_times"] for h in histories],
    "error time",
)


# ------------------------------------------------------------
# Verify common error-evaluation checkpoints.
# ------------------------------------------------------------

error_iterations_all = [
    np.asarray(
        h["error_iterations"],
        dtype=int,
    )
    for h in histories
]

reference_error_iterations = error_iterations_all[0]

if not all(
    np.array_equal(
        it,
        reference_error_iterations,
    )
    for it in error_iterations_all[1:]
):
    raise RuntimeError(
        "Relative-error checkpoint iterations differ across seeds."
    )


loss_iterations = np.arange(
    1,
    loss_stack.shape[1] + 1,
)

error_iterations = reference_error_iterations


# ------------------------------------------------------------
# Median and interquartile range across seeds.
#
# Q1 = 25th percentile.
# Q3 = 75th percentile.
#
# [Q1,Q3] contains the middle 50% of random-seed runs.
# ------------------------------------------------------------

loss_median, loss_q25, loss_q75 = _median_iqr(
    loss_stack
)

error_median, error_q25, error_q75 = _median_iqr(
    error_stack
)


# ------------------------------------------------------------
# Wall-clock times differ slightly between seeds.
# Use the median cumulative time at each common error checkpoint.
# ------------------------------------------------------------

time_median = np.median(
    time_stack,
    axis=0,
)

time_q25, time_q75 = np.percentile(
    time_stack,
    [25.0, 75.0],
    axis=0,
)


# ------------------------------------------------------------
# Minimalistic 1 x 3 figure: median (solid) and IQR band (shaded).
# No titles, no legends, no suptitle.
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.6, 2.7),
    constrained_layout=True,
)

line_color = "tab:blue"

panels = (
    (loss_iterations, loss_median, loss_q25, loss_q75,
     "iteration", "Loss"),
    (error_iterations, error_median, error_q25, error_q75,
     "iteration", r"Relative $L^2$ error"),
    (time_median, error_median, error_q25, error_q75,
     "Wall clock time [s]", r"Relative $L^2$ error"),
)

for ax, (x, median, q25, q75, xlabel, ylabel) in zip(axes, panels):
    ax.fill_between(
        x,
        q25,
        q75,
        color=line_color,
        alpha=0.20,
        linewidth=0.0,
    )
    ax.plot(
        x,
        median,
        color=line_color,
        linewidth=2.0,
    )
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, which="major", linewidth=0.4, alpha=0.20)
    ax.grid(True, which="minor", linewidth=0.25, alpha=0.10)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    ax.tick_params(which="both", direction="out")


# ------------------------------------------------------------
# Export figure.
# ------------------------------------------------------------

figure_path = (
    result_dir
    / "tt_heat_adam_10seed_median_iqr.png"
)

figure_pdf_path = (
    result_dir
    / "tt_heat_adam_10seed_median_iqr.pdf"
)

fig.savefig(
    figure_path,
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    figure_pdf_path,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# One row per TT run.
# ------------------------------------------------------------

run_table = pd.DataFrame([
    ablation_results[seed]["record"]
    for seed in seeds
])

run_table["format"] = (
    run_table["format"].str.upper()
)

run_table["ms_per_iteration"] = (
    1e3 * run_table["t_per_iter"]
)


run_table = run_table.rename(
    columns={
        "active_rank": "model_rank",
        "active_width": "width",
        "active_hidden": "hidden_layers",
        "P_axis": "parameters_per_core",
        "P": "total_parameters",
        "loss_final": "final_loss",
        "err_final": "final_rel_L2",
        "err_best": "best_rel_L2",
        "loss_pde_final": "final_PDE_loss",
        "loss_boundary_final": "final_boundary_loss",
        "loss_initial_final": "final_initial_loss",
        "loss_boundary_weighted_final":
            "final_boundary_weighted_loss",
        "loss_initial_weighted_final":
            "final_initial_weighted_loss",
    }
)


run_columns = [
    "format",
    "seed",
    "solver_used",
    "network_shape",
    "model_rank",
    "width",
    "hidden_layers",
    "parameters_per_core",
    "total_parameters",
    "N_eff",
    "tt_init_mode",
    "tt_identity_epsilon",
    "tt_global_scale",
    "boundary_weight",
    "initial_weight",
    "boundary_target",
    "iterations",
    "error_every",
    "final_loss",
    "final_PDE_loss",
    "final_boundary_loss",
    "final_initial_loss",
    "final_boundary_weighted_loss",
    "final_initial_weighted_loss",
    "final_rel_L2",
    "best_rel_L2",
    "t_run",
    "ms_per_iteration",
]


run_table = (
    run_table[run_columns]
    .sort_values("seed")
    .reset_index(drop=True)
)


display(
    run_table.style.format({
        "final_loss": "{:.4e}",
        "final_PDE_loss": "{:.4e}",
        "final_boundary_loss": "{:.4e}",
        "final_initial_loss": "{:.4e}",
        "final_boundary_weighted_loss": "{:.4e}",
        "final_initial_weighted_loss": "{:.4e}",
        "final_rel_L2": "{:.4e}",
        "best_rel_L2": "{:.4e}",
        "t_run": "{:.3f}",
        "ms_per_iteration": "{:.2f}",
    })
)


# ------------------------------------------------------------
# Scalar robustness summary over the ten seeds.
#
# We report:
#     median
#     Q1 = 25th percentile
#     Q3 = 75th percentile
#     IQR = Q3 - Q1
# ------------------------------------------------------------

final_error_q25 = (
    run_table["final_rel_L2"].quantile(0.25)
)

final_error_q75 = (
    run_table["final_rel_L2"].quantile(0.75)
)

final_loss_q25 = (
    run_table["final_loss"].quantile(0.25)
)

final_loss_q75 = (
    run_table["final_loss"].quantile(0.75)
)

time_per_it_q25 = (
    run_table["ms_per_iteration"].quantile(0.25)
)

time_per_it_q75 = (
    run_table["ms_per_iteration"].quantile(0.75)
)


aggregate_table = pd.DataFrame([{

    "seeds":
        len(seeds),

    "final_rel_L2_median":
        run_table["final_rel_L2"].median(),

    "final_rel_L2_q25":
        final_error_q25,

    "final_rel_L2_q75":
        final_error_q75,

    "final_rel_L2_IQR":
        final_error_q75 - final_error_q25,

    "best_rel_L2_median":
        run_table["best_rel_L2"].median(),

    "final_loss_median":
        run_table["final_loss"].median(),

    "final_loss_q25":
        final_loss_q25,

    "final_loss_q75":
        final_loss_q75,

    "final_loss_IQR":
        final_loss_q75 - final_loss_q25,

    "ms_per_iteration_median":
        run_table["ms_per_iteration"].median(),

    "ms_per_iteration_q25":
        time_per_it_q25,

    "ms_per_iteration_q75":
        time_per_it_q75,

    "ms_per_iteration_IQR":
        time_per_it_q75 - time_per_it_q25,
}])


display(
    aggregate_table.style.format({

        "final_rel_L2_median": "{:.4e}",
        "final_rel_L2_q25": "{:.4e}",
        "final_rel_L2_q75": "{:.4e}",
        "final_rel_L2_IQR": "{:.4e}",

        "best_rel_L2_median": "{:.4e}",

        "final_loss_median": "{:.4e}",
        "final_loss_q25": "{:.4e}",
        "final_loss_q75": "{:.4e}",
        "final_loss_IQR": "{:.4e}",

        "ms_per_iteration_median": "{:.2f}",
        "ms_per_iteration_q25": "{:.2f}",
        "ms_per_iteration_q75": "{:.2f}",
        "ms_per_iteration_IQR": "{:.2f}",
    })
)


# ------------------------------------------------------------
# Aggregate histories used in the figure.
# ------------------------------------------------------------

loss_history_table = pd.DataFrame({

    "iteration":
        loss_iterations,

    "loss_median":
        loss_median,

    "loss_q25":
        loss_q25,

    "loss_q75":
        loss_q75,

    "loss_IQR":
        loss_q75 - loss_q25,
})


error_history_table = pd.DataFrame({

    "iteration":
        error_iterations,

    "median_wall_time_s":
        time_median,

    "wall_time_q25_s":
        time_q25,

    "wall_time_q75_s":
        time_q75,

    "relative_l2_median":
        error_median,

    "relative_l2_q25":
        error_q25,

    "relative_l2_q75":
        error_q75,

    "relative_l2_IQR":
        error_q75 - error_q25,
})


# ------------------------------------------------------------
# Export CSV files.
# ------------------------------------------------------------

run_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_runs.csv"
)

aggregate_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_summary_median_iqr.csv"
)

loss_history_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_loss_history_median_iqr.csv"
)

error_history_csv_path = (
    result_dir
    / "tt_heat_adam_10seed_error_history_median_iqr.csv"
)

json_path = (
    result_dir
    / "tt_heat_adam_10seed_histories_median_iqr.json"
)


run_table.to_csv(
    run_csv_path,
    index=False,
)

aggregate_table.to_csv(
    aggregate_csv_path,
    index=False,
)

loss_history_table.to_csv(
    loss_history_csv_path,
    index=False,
)

error_history_table.to_csv(
    error_history_csv_path,
    index=False,
)


# ------------------------------------------------------------
# Complete JSON export.
# ------------------------------------------------------------

payload = {

    "settings": {

        "problem":
            "10+1 dimensional heat equation with "
            "soft spatial-boundary and initial residuals",

        "format":
            "tt",

        "spatial_dim":
            int(SPATIAL_DIM),

        "diffusivity":
            float(DIFFUSIVITY),

        "boundary_weight":
            float(SOFT_BOUNDARY_WEIGHT),

        "initial_weight":
            float(SOFT_INITIAL_WEIGHT),

        "boundary_target":
            BOUNDARY_TARGET,

        "seeds":
            seeds,

        "iterations":
            int(ABLATION_ITERATIONS),

        "error_every":
            int(ABLATION_ERROR_EVERY),

        "tt": {

            "rank":
                int(TT_RANK),

            "width":
                int(TT_WIDTH),

            "hidden":
                int(TT_HIDDEN),

            "solver":
                ABLATION_TT_SOLVER,

            "init_mode":
                TT_INIT_MODE,

            "identity_epsilon":
                float(TT_IDENTITY_EPS),

            "global_scale":
                bool(TT_GLOBAL_SCALE),
        },
    },


    "aggregate_history": {

        "statistics":
            "median with interquartile range "
            "(25th--75th percentiles)",

        "loss_iteration":
            loss_iterations.tolist(),

        "loss_median":
            loss_median.tolist(),

        "loss_q25":
            loss_q25.tolist(),

        "loss_q75":
            loss_q75.tolist(),

        "loss_IQR":
            (loss_q75 - loss_q25).tolist(),

        "error_iteration":
            error_iterations.tolist(),

        "median_wall_time_s":
            time_median.tolist(),

        "wall_time_q25_s":
            time_q25.tolist(),

        "wall_time_q75_s":
            time_q75.tolist(),

        "relative_l2_median":
            error_median.tolist(),

        "relative_l2_q25":
            error_q25.tolist(),

        "relative_l2_q75":
            error_q75.tolist(),

        "relative_l2_IQR":
            (error_q75 - error_q25).tolist(),
    },


    "runs": {

        str(seed): {

            "record":
                ablation_results[seed]["record"],

            "history":
                _json_history(
                    ablation_results[seed]["history"]
                ),
        }

        for seed in seeds
    },
}


with open(
    json_path,
    "w",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Report saved files.
# ------------------------------------------------------------

print(
    f"saved PNG figure:              {figure_path}"
)

print(
    f"saved PDF figure:              {figure_pdf_path}"
)

print(
    f"saved per-run table:           {run_csv_path}"
)

print(
    f"saved scalar summary:          {aggregate_csv_path}"
)

print(
    f"saved median/IQR loss history: {loss_history_csv_path}"
)

print(
    f"saved median/IQR error history:{error_history_csv_path}"
)

print(
    f"saved all TT histories:        {json_path}"
)